In [1]:
import json
import os
import subprocess
import itertools
from datetime import datetime
import pandas as pd
from pathlib import Path
import sys
import pickle as pl
import torch
import itertools

In [2]:
env = os.environ.copy()
env['MKL_SERVICE_FORCE_INTEL'] = '1'
env['MKL_THREADING_LAYER'] = 'GNU'
env['OMP_NUM_THREADS'] = '1'

In [3]:
# parameter_grid_1 = {
#     "n_total": [5000],
#     "n_finetune": [2500],
#     "model_name": ["bert-base-uncased"],
#     "max_length": [256],
#     "num_labels": [6],
#     "batch_size": [32],
#     "learning_rate": [1e-2,0.003],
#     "num_epochs": [2],
#     "K": [15],
#     "lambda_min": [0.05],
#     "lambda_max": [0.95],
#     "interpolation": ["linear"],
#     "optimizer": ["Adam"],
#     "dataset": ["emotion"],
#     "proportionArr": [[0.15,0.15,0.3,0.35,0.05]]
# }

In [4]:
proportion_arr = [[0.3, 0.05, 0.3, 0.35],[0.05, 0.3, 0.3, 0.35],[0.3, 0.3, 0.3, 0.1]]

In [5]:
step = 0.05
total_const = 0.3 + 0.3   # sum of the two fixed proportions
remaining_sum = 1.0 - total_const  # here = 0.4

values = [round(step * i, 2) for i in range(int(remaining_sum / step) + 1)]

vectors = []

indices = [0, 1, 2, 3]
for i, j in itertools.combinations(indices, 2):  # all distinct pairs to keep constant
    for x in values:
        y = round(remaining_sum - x, 2)
        v = [0.0]*4
        v[i] = 0.3
        v[j] = 0.3
        # remaining two positions:
        k, l = [idx for idx in indices if idx not in (i, j)]
        v[k] = x
        v[l] = y
        vectors.append(v)


In [6]:
# 0.000006 - stable learning rate

parameter_grid_2 = {
    "n_total": [5000],
    "n_finetune": [2500],
    "model_name": ["bert-base-uncased"],
    "max_length": [256],
    "num_labels": [4],
    "batch_size": [32],
    "learning_rate": [0.000006],
    "num_epochs": [2],
    "K": [15],
    "lambda_min": [0.05],
    "lambda_max": [0.95],
    "interpolation": ["model_baseline"],
    "optimizer": ["Adam"],
    "dataset": ["ag_news"],
    "proportionArr": proportion_arr
}

# 0.000006, 0.00001

# # to be fixed
# [0.1,0.7,0.1,0.1] 

# # done 
# [0.25,0.25,0.25,0.25]
# [0.3,0.1,0.3,0.3]


    # "learning_rate": [0.000006,0.000003],



In [7]:
# parameter_grid_3 = {
#     "n_total": [5000],
#     "n_finetune": [2500],
#     "model_name": ["bert-base-uncased"],
#     "max_length": [256],
#     "num_labels": [5],
#     "batch_size": [32],
#     "learning_rate": [0.001,0.003],
#     "num_epochs": [2],
#     "K": [15],
#     "lambda_min": [0.05],
#     "lambda_max": [0.95],
#     "interpolation": ["linear"],
#     "optimizer": ["Adam"],
#     "dataset": ["yelp_review_full"],
#     "proportionArr": [[0.27,0.1,0.27,0.1,0.26],]
    
    
    # # to be fixed
    # [0.1,0.1,0.6,0.1,0.1]
    # [0.05,0.1,0.05,0.7,0.1]
    
    
    # # done 
    # [0.2,0.2,0.2,0.2,0.2]


In [8]:
parameter_grids = []
parameter_grids.append(parameter_grid_2)
# parameter_grids.append(parameter_grid_3)

In [9]:
# Or define specific combinations
specific_configs = []

In [10]:
def generate_all_combinations(param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = []
    
    for combination in itertools.product(*values):
        config = dict(zip(keys, combination))
        combinations.append(config)
    
    return combinations

def create_config_file(config, experiment_path):
    config['experiment_name'] = experiment_path
    config_path = experiment_path + ".json"
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    print(f"Created config file: {config_path}")

def run_experiment(config, experiment_name):
    try:
        # Create config file
        create_config_file(config, f"{experiment_name}")
        
        print(f"\nRunning experiment: {experiment_name}")
        print(f"Config: {config}")
        
        # result = subprocess.run([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"], capture_output=True, text=True,shell=True)
        result = subprocess.call([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"],env=env)
        
        if result == 0:
            print(f"✅ Experiment {experiment_name} completed successfully")
        else:
            print(f"❌ Experiment {experiment_name} failed")
            # print("Error:", result.stderr)
        
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": result == 0,
            # "stdout": result.stdout,
            # "stderr": result.stderr
        }
        
    except Exception as e:
        print(f"❌ Exception in {experiment_name}: {str(e)}")
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": False,
            "error": str(e)
        }

In [11]:
configs_to_run = generate_all_combinations(parameter_grids[0])
len(configs_to_run)

3

In [ ]:
# Choose experiment mode
USE_GRID_SEARCH = True # Set to True for grid search, False for specific configs

if USE_GRID_SEARCH:
    for parameter_grid in parameter_grids:
        configs_to_run = generate_all_combinations(parameter_grid)
        print(f"Total configurations to run: {len(configs_to_run)}")
        
        # Run all experiments
        results = []
        for i, config in enumerate(configs_to_run):
            experiment_name = f"{config['dataset']}_{config['learning_rate']}_{config['proportionArr']}"
            result = run_experiment(config, experiment_name)
            results.append(result)

        # Summary
        successful = sum(1 for r in results if r["success"])
        print(f"\n{'='*50}")
        print(f"EXPERIMENT SUMMARY")
        print(f"{'='*50}")
        print(f"Total experiments: {len(results)}")
        print(f"Successful: {successful}")
        print(f"Failed: {len(results) - successful}")
        
else:
    configs_to_run = specific_configs

Total configurations to run: 3
Created config file: ag_news_6e-06_[0.3, 0.05, 0.3, 0.35].json

Running experiment: ag_news_6e-06_[0.3, 0.05, 0.3, 0.35]
Config: {'n_total': 5000, 'n_finetune': 2500, 'model_name': 'bert-base-uncased', 'max_length': 256, 'num_labels': 4, 'batch_size': 32, 'learning_rate': 6e-06, 'num_epochs': 2, 'K': 15, 'lambda_min': 0.05, 'lambda_max': 0.95, 'interpolation': 'model_baseline', 'optimizer': 'Adam', 'dataset': 'ag_news', 'proportionArr': [0.3, 0.05, 0.3, 0.35], 'experiment_name': 'ag_news_6e-06_[0.3, 0.05, 0.3, 0.35]'}


2025-12-13 18:33:26.458293: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-13 18:33:26.475654: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-13 18:33:26.475674: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-13 18:33:26.476220: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-13 18:33:26.479128: I tensorflow/core/platform/cpu_feature_guar

Loading configuration from: ag_news_6e-06_[0.3, 0.05, 0.3, 0.35].json
✓ Configuration loaded successfully

Output directory created: ag_news_6e-06_[0.3, 0.05, 0.3, 0.35]

STEP 1: Loading the ag_news Dataset
Full AG News training set size: 120000
Selected subset D with 5000 samples
Label 0....samples needed 750.....needed proportion: 0.3....actual proportion: 0.3
Label 1....samples needed 125.....needed proportion: 0.05....actual proportion: 0.05
Label 2....samples needed 750.....needed proportion: 0.3....actual proportion: 0.3
Label 3....samples needed 875.....needed proportion: 0.35....actual proportion: 0.35
Size of the computed finetuning set: 2500 
Fine-tuning set expected size: 2500
Size of the fixed/updated finetuning set: 2500 
Selected fine-tuning subset D' with 2500 samples

Dataset Statistics:
  Total samples |D|: 5000
  Fine-tuning samples |D'|: 2500
  Ratio |D'|/|D|: 50.00%
  Indicator vector sum: 2500
  Positive class ratio: 50.00%

✓ Dataset info saved to ag_news_6e-06_[0

Epoch 1/2:  11%|█▏        | 9/79 [00:03<00:20,  3.41it/s, loss=1.19]

In [ ]:
# When a parent process starts a child process via subprocess, the two are separate entities with their own memory space.
# The parent process is not notified in real-time about the filesystem modifications the child process is making.